In [35]:

%pip install -q pandas numpy pillow opencv-python-headless pytesseract pymupdf scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:

from pathlib import Path
import os, re, io, json, sqlite3, hashlib, unicodedata
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
import cv2
import fitz
import pytesseract
from pytesseract import Output
from IPython.display import display

BASE = Path("proyecto_documentos")
ENTRADA = BASE / "documentos"
SALIDA = BASE / "resultados"

ENTRADA.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)

EXTENSIONES = {
    ".pdf", ".jpg", ".jpeg", ".png",
    ".tif", ".tiff", ".bmp", ".webp"
}

RUTA_TESSERACT = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    if os.name == "nt" else None
)

if RUTA_TESSERACT and Path(RUTA_TESSERACT).exists():
    pytesseract.pytesseract.tesseract_cmd = RUTA_TESSERACT

try:
    VERSION_OCR = str(pytesseract.get_tesseract_version())
    IDIOMAS = set(pytesseract.get_languages(config=""))
    OCR_OK = True
except Exception:
    VERSION_OCR = "no disponible"
    IDIOMAS = set()
    OCR_OK = False

if {"spa", "eng"}.issubset(IDIOMAS):
    IDIOMA_OCR = "spa+eng"
elif "spa" in IDIOMAS:
    IDIOMA_OCR = "spa"
elif "eng" in IDIOMAS:
    IDIOMA_OCR = "eng"
else:
    IDIOMA_OCR = ""

MAX_PAGINAS_PDF = 200
MIN_CARACTERES_TEXTO = 35
DPI_PDF = 220

print("Carpeta de entrada:", ENTRADA.resolve())
print("Carpeta de resultados:", SALIDA.resolve())
print("OCR:", VERSION_OCR)
print("Idioma:", IDIOMA_OCR or "ninguno")


Carpeta de entrada: C:\Users\mcgre\Desktop\yaroy\proyecto_documentos\documentos
Carpeta de resultados: C:\Users\mcgre\Desktop\yaroy\proyecto_documentos\resultados
OCR: 5.5.3.20260724
Idioma: eng


In [37]:

def limpiar_texto(texto):
    texto = (texto or "").replace("\x00", " ")
    return "\n".join(
        re.sub(r"[ \t]+", " ", linea).strip()
        for linea in texto.splitlines()
    ).strip()


def normalizar(texto):
    return "".join(
        c for c in unicodedata.normalize(
            "NFKD", (texto or "").lower()
        )
        if not unicodedata.combining(c)
    )


def estrategias_imagen(imagen):
    rgb = np.array(imagen.convert("RGB"))
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    alto, ancho = gris.shape

    if max(alto, ancho) < 1800:
        factor = min(2.0, 1800 / max(alto, ancho))
        gris = cv2.resize(
            gris, None,
            fx=factor, fy=factor,
            interpolation=cv2.INTER_CUBIC
        )

    gris = cv2.fastNlMeansDenoising(gris, h=8)

    adaptive = cv2.adaptiveThreshold(
        gris, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 35, 11
    )

    return [gris, adaptive]


def ocr_imagen(imagen):
    if not OCR_OK or not IDIOMA_OCR:
        return "", 0.0, "ocr_no_disponible"

    candidatos = []

    for n_imagen, img in enumerate(estrategias_imagen(imagen)):
        for psm in (3, 6):
            try:
                config = f"--oem 3 --psm {psm}"

                datos = pytesseract.image_to_data(
                    img,
                    lang=IDIOMA_OCR,
                    config=config,
                    output_type=Output.DICT
                )

                palabras = []
                valores = []

                for palabra, conf in zip(
                    datos["text"], datos["conf"]
                ):
                    if palabra and palabra.strip():
                        palabras.append(palabra.strip())
                        try:
                            if float(conf) >= 0:
                                valores.append(float(conf))
                        except (ValueError, TypeError):
                            pass

                texto = limpiar_texto(
                    pytesseract.image_to_string(
                        img, lang=IDIOMA_OCR, config=config
                    )
                )

                confianza = (
                    round(float(np.mean(valores)), 1)
                    if valores else 0.0
                )

                puntuacion = (
                    confianza + min(len(palabras), 100) * 0.03
                )

                candidatos.append((
                    puntuacion, texto, confianza,
                    f"ocr_{n_imagen}_psm{psm}"
                ))

            except Exception:
                continue

    if not candidatos:
        return "", 0.0, "error_ocr"

    _, texto, confianza, metodo = max(
        candidatos,
        key=lambda item: (item[0], len(item[1]))
    )

    return texto, confianza, metodo


def leer_documento(ruta):
    ruta = Path(ruta)
    paginas = []
    avisos = []

    if ruta.suffix.lower() == ".pdf":
        with fitz.open(ruta) as doc:
            if doc.needs_pass:
                raise ValueError(
                    "PDF protegido con contraseña"
                )

            if len(doc) > MAX_PAGINAS_PDF:
                avisos.append(
                    f"Solo se procesaron {MAX_PAGINAS_PDF} "
                    f"de {len(doc)} páginas"
                )

            for i in range(min(len(doc), MAX_PAGINAS_PDF)):
                pagina = doc[i]

                texto = limpiar_texto(
                    pagina.get_text("text", sort=True)
                )

                metodo = "texto_pdf"
                confianza = None

                cantidad = len(re.sub(r"\s", "", texto))

                if cantidad < MIN_CARACTERES_TEXTO:
                    try:
                        pix = pagina.get_pixmap(
                            matrix=fitz.Matrix(
                                DPI_PDF / 72,
                                DPI_PDF / 72
                            ),
                            alpha=False
                        )

                        imagen = Image.open(
                            io.BytesIO(pix.tobytes("png"))
                        )

                        texto_ocr, confianza, metodo_ocr = (
                            ocr_imagen(imagen)
                        )

                        if len(texto_ocr) > len(texto):
                            texto = texto_ocr
                            metodo = metodo_ocr

                    except Exception as e:
                        avisos.append(
                            f"Página {i+1}: error OCR ({e})"
                        )

                paginas.append({
                    "pagina": i + 1,
                    "texto": texto,
                    "metodo": metodo,
                    "confianza_ocr": confianza
                })

    else:
        with Image.open(ruta) as im:
            n_frames = min(
                getattr(im, "n_frames", 1),
                MAX_PAGINAS_PDF
            )

            for i in range(n_frames):
                im.seek(i)

                imagen = ImageOps.exif_transpose(
                    im.copy().convert("RGB")
                )

                texto, confianza, metodo = ocr_imagen(
                    imagen
                )

                paginas.append({
                    "pagina": i + 1,
                    "texto": texto,
                    "metodo": metodo,
                    "confianza_ocr": confianza
                })

    return paginas, avisos


In [38]:

CATEGORIAS = {
    "Factura": {
        "factura": 4,
        "subtotal": 2,
        "iva": 1,
        "factura de venta": 4
    },
    "Orden de compra": {
        "orden de compra": 6,
        "orden de pedido": 3,
        "precio unitario": 2
    },
    "Recibo": {
        "recibo": 4,
        "recibi": 2,
        "recibo de caja": 5
    },
    "Contrato": {
        "contrato": 5,
        "clausula": 2,
        "contratante": 2,
        "contratista": 2
    },
    "Formulario": {
        "formulario": 5,
        "fecha de nacimiento": 2,
        "apellidos": 1
    },
    "Comprobante": {
        "comprobante": 4,
        "transaccion": 2,
        "transferencia": 1,
        "referencia": 1
    }
}


def clasificar(texto):
    t = normalizar(texto)

    puntajes = {
        categoria: sum(
            peso
            for termino, peso in indicadores.items()
            if termino in t
        )
        for categoria, indicadores in CATEGORIAS.items()
    }

    orden = sorted(
        puntajes,
        key=puntajes.get,
        reverse=True
    )

    primero = puntajes[orden[0]]
    segundo = puntajes[orden[1]]

    if primero < 3 or primero == segundo:
        return "Revisión necesaria", puntajes

    return orden[0], puntajes


def buscar(texto, patrones):
    for patron in patrones:
        coincidencia = re.search(
            patron, texto, re.I | re.M
        )

        if coincidencia:
            valor = coincidencia.group(1).strip()
            valor = valor.strip(" -:#")

            if valor:
                return valor[:200]

    return None


def extraer_campos(texto):
    numero = buscar(texto, [
        r"^\s*(?:factura|recibo|comprobante|orden(?: de compra)?)\s*(?:de venta)?\s*(?:n[°ºo.]?\s*)?[:#-]?\s*([A-Z0-9][A-Z0-9\-/]{2,})\s*$",
        r"^\s*(?:n[úu]mero de (?:factura|documento)|folio|factura n[°ºo.]?)\s*[:#-]\s*([A-Z0-9\-/]{3,})"
    ])

    fecha = buscar(texto, [
        r"\b(\d{4}[-/]\d{1,2}[-/]\d{1,2})\b",
        r"\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b"
    ])

    nit = buscar(texto, [
        r"\bNIT\s*[:#-]?\s*(\d[\d.\- ]{4,18}\d)\b"
    ])

    totales = re.findall(
        r"^\s*(?:TOTAL(?: A PAGAR)?|VALOR TOTAL|IMPORTE TOTAL)\s*[:$ ]+\s*\$?\s*([\d][\d., ]{0,20})\s*$",
        texto,
        re.I | re.M
    )

    total = totales[-1].strip() if totales else None

    cliente = buscar(texto, [
        r"^\s*(?:cliente|raz[oó]n social del cliente)\s*[:\-]\s*(.{3,100})$"
    ])

    proveedor = buscar(texto, [
        r"^\s*(?:proveedor|emisor)\s*[:\-]\s*(.{3,100})$"
    ])

    return {
        "numero_documento": numero,
        "fecha": fecha,
        "nit": nit,
        "total": total,
        "cliente": cliente,
        "proveedor": proveedor
    }


texto_demo = """
FACTURA DE VENTA
NIT: 900123456-7
Cliente: Empresa Ejemplo
Fecha: 10/09/2026
Subtotal: 100000
IVA: 19000
TOTAL: 119000
"""

print("Categoría:", clasificar(texto_demo)[0])
print("Campos:", extraer_campos(texto_demo))


Categoría: Factura
Campos: {'numero_documento': None, 'fecha': '10/09/2026', 'nit': '900123456-7', 'total': '119000', 'cliente': 'Empresa Ejemplo', 'proveedor': None}


In [39]:

DB = BASE / "documentos.db"


def sha256_archivo(ruta):
    h = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024), b""
        ):
            h.update(bloque)

    return h.hexdigest()


def preparar_db(con):
    con.executescript("""
    CREATE TABLE IF NOT EXISTS documentos_v2 (
      ruta TEXT PRIMARY KEY,
      hash_sha256 TEXT,
      categoria TEXT,
      puntajes_json TEXT,
      numero_documento TEXT,
      fecha TEXT,
      nit TEXT,
      total TEXT,
      cliente TEXT,
      proveedor TEXT,
      num_paginas INTEGER,
      estado TEXT,
      avisos TEXT,
      texto_extraido TEXT,
      actualizado_en TEXT DEFAULT CURRENT_TIMESTAMP
    );

    CREATE TABLE IF NOT EXISTS paginas_v2 (
      ruta TEXT,
      pagina INTEGER,
      metodo TEXT,
      confianza_ocr REAL,
      texto TEXT,
      PRIMARY KEY(ruta, pagina)
    );
    """)


def procesar_lote():
    rutas = sorted(
        p for p in ENTRADA.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTENSIONES
    )

    print(f"Documentos encontrados: {len(rutas)}")

    with sqlite3.connect(DB) as con:
        preparar_db(con)

        for idx, ruta in enumerate(rutas, 1):
            relativa = ruta.relative_to(ENTRADA).as_posix()

            try:
                paginas, avisos = leer_documento(ruta)

                texto = "\n\n".join(
                    f"[PÁGINA {p['pagina']}]\n{p['texto']}"
                    for p in paginas
                )

                categoria, puntajes = clasificar(texto)
                campos = extraer_campos(texto)

                estado = (
                    "ok"
                    if any(p["texto"].strip() for p in paginas)
                    else "sin_texto"
                )

                if estado == "sin_texto":
                    avisos.append(
                        "Sin texto recuperable"
                    )

                registro = (
                    relativa,
                    sha256_archivo(ruta),
                    categoria,
                    json.dumps(puntajes, ensure_ascii=False),
                    campos["numero_documento"],
                    campos["fecha"],
                    campos["nit"],
                    campos["total"],
                    campos["cliente"],
                    campos["proveedor"],
                    len(paginas),
                    estado,
                    "; ".join(avisos),
                    texto
                )

                con.execute("""
                    INSERT OR REPLACE INTO documentos_v2
                    (
                        ruta, hash_sha256, categoria,
                        puntajes_json, numero_documento,
                        fecha, nit, total, cliente,
                        proveedor, num_paginas,
                        estado, avisos, texto_extraido,
                        actualizado_en
                    )
                    VALUES (
                        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                        ?, ?, ?, ?, CURRENT_TIMESTAMP
                    )
                """, registro)

                con.execute(
                    "DELETE FROM paginas_v2 WHERE ruta=?",
                    (relativa,)
                )

                con.executemany("""
                    INSERT INTO paginas_v2
                    (
                        ruta, pagina, metodo,
                        confianza_ocr, texto
                    )
                    VALUES (?, ?, ?, ?, ?)
                """, [
                    (
                        relativa,
                        p["pagina"],
                        p["metodo"],
                        p["confianza_ocr"],
                        p["texto"]
                    )
                    for p in paginas
                ])

            except Exception as e:
                estado = "error"

                con.execute("""
                    INSERT OR REPLACE INTO documentos_v2
                    (
                        ruta, hash_sha256, categoria,
                        estado, avisos, num_paginas,
                        actualizado_en
                    )
                    VALUES (
                        ?, ?, 'Revisión necesaria',
                        'error', ?, 0, CURRENT_TIMESTAMP
                    )
                """, (
                    relativa,
                    sha256_archivo(ruta),
                    str(e)
                ))

            con.commit()

            print(
                f"[{idx}/{len(rutas)}] "
                f"{estado}: {relativa}"
            )

        df = pd.read_sql_query(
            "SELECT * FROM documentos_v2 ORDER BY ruta",
            con
        )

        df_paginas = pd.read_sql_query(
            "SELECT * FROM paginas_v2 ORDER BY ruta,pagina",
            con
        )

    columnas_ocultas = [
        "texto_extraido",
        "puntajes_json",
        "hash_sha256"
    ]

    df.drop(
        columns=columnas_ocultas,
        errors="ignore"
    ).to_csv(
        SALIDA / "resumen_documentos.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df_paginas.to_csv(
        SALIDA / "texto_por_pagina.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Resultados:",
        (SALIDA / "resumen_documentos.csv").resolve()
    )

    return df, df_paginas


df, df_paginas = procesar_lote()

display(
    df.drop(
        columns=[
            "texto_extraido",
            "puntajes_json",
            "hash_sha256"
        ],
        errors="ignore"
    )
)


Documentos encontrados: 7
[1/7] ok: 566w-7tfxbmQkCPc.png
[2/7] ok: Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2).pdf
[3/7] ok: Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2)_pagina_1.png
[4/7] ok: Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg
[5/7] ok: image.png
[6/7] ok: modelo-de-recibo-es-puro-750px.png
[7/7] ok: orden-de-compra-formato.png
Resultados: C:\Users\mcgre\Desktop\yaroy\proyecto_documentos\resultados\resumen_documentos.csv


,ruta,categoria,numero_documento,fecha,nit,total,cliente,proveedor,num_paginas,estado,avisos,actualizado_en
0,566w-7tfxbmQkCPc.png,Recibo,None,NaN,None,None,None,None,1,ok,,2026-10-08 16:26:31
1,Documento A4 Contrato de servicios Asistente v...,Contrato,None,NaN,None,None,None,None,1,ok,,2026-10-08 16:26:32
2,Documento A4 Contrato de servicios Asistente v...,Revisión necesaria,None,NaN,None,None,None,None,1,ok,,2026-10-08 16:26:40
3,Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg,Factura,None,21/02/2018,None,None,None,None,1,ok,,2026-10-08 16:26:46
4,image.png,Formulario,None,NaN,None,None,None,None,1,ok,,2026-10-08 16:26:51
5,modelo-de-recibo-es-puro-750px.png,Recibo,None,29/01/2019,None,None,None,None,1,ok,,2026-10-08 16:26:56
6,orden-de-compra-formato.png,Revisión necesaria,None,NaN,None,None,None,None,1,ok,,2026-10-08 16:27:01


In [40]:

import fitz
from pathlib import Path

carpeta = Path("proyecto_documentos/entrenamiento")

for pdf in carpeta.rglob("*.pdf"):
    try:
        with fitz.open(pdf) as documento:
            for numero, pagina in enumerate(documento):
                pix = pagina.get_pixmap(
                    matrix=fitz.Matrix(2, 2),
                    alpha=False
                )

                destino = pdf.with_name(
                    f"{pdf.stem}_pagina_{numero + 1}.png"
                )

                pix.save(str(destino))
                print("Convertido:", destino)

    except Exception as error:
        print("Error:", pdf, error)

print("Conversión terminada")

Convertido: proyecto_documentos\entrenamiento\Contratos\Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2)_pagina_1.png
Conversión terminada


In [41]:

# Instalar previamente si hace falta:
# %pip install tensorflow

ENTRENAR_MODELO = True
EPOCAS_MAXIMAS = 50

CARPETA_ENTRENAMIENTO = BASE / "entrenamiento"

if ENTRENAR_MODELO:
    import tensorflow as tf
    from tensorflow import keras

    if not CARPETA_ENTRENAMIENTO.exists():
        raise FileNotFoundError(
            f"Crea carpetas por categoría en "
            f"{CARPETA_ENTRENAMIENTO}"
        )

    train_ds = keras.utils.image_dataset_from_directory(
        CARPETA_ENTRENAMIENTO,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=(224, 224),
        batch_size=16
    )

    val_ds = keras.utils.image_dataset_from_directory(
        CARPETA_ENTRENAMIENTO,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=(224, 224),
        batch_size=16
    )

    nombres_clases = train_ds.class_names

    preprocesador = keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    preprocesador.trainable = False

    entrada = keras.Input(shape=(224, 224, 3))

    x = keras.layers.RandomRotation(0.03)(entrada)
    x = keras.layers.RandomZoom(0.1)(x)

    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = preprocesador(x, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)

    salida = keras.layers.Dense(
        len(nombres_clases),
        activation="softmax"
    )(x)

    modelo = keras.Model(entrada, salida)

    modelo.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3
        )
    ]

    historial = modelo.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCAS_MAXIMAS,
        callbacks=callbacks
    )

    modelo.save(
        str(SALIDA / "modelo_documentos.keras")
    )

    (SALIDA / "clases_modelo.json").write_text(
        json.dumps(
            nombres_clases,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        "Validación:",
        modelo.evaluate(val_ds, verbose=0)
    )

else:
    print(
        "Entrenamiento desactivado. "
        "Para activarlo, prepara datos etiquetados "
        "y cambia ENTRENAR_MODELO a True."
    )


Found 6 files belonging to 6 classes.


Using 5 files for training.
Found 6 files belonging to 6 classes.
Using 1 files for validation.
Epoch 1/50


c:\Users\mcgre\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.4000 - loss: 1.8066 - val_accuracy: 0.0000e+00 - val_loss: 2.9523 - learning_rate: 0.0010
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.4000 - loss: 1.5766 - val_accuracy: 0.0000e+00 - val_loss: 3.1145 - learning_rate: 0.0010
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.4000 - loss: 1.6440 - val_accuracy: 0.0000e+00 - val_loss: 3.2676 - learning_rate: 0.0010
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.8000 - loss: 1.1881 - val_accuracy: 0.0000e+00 - val_loss: 3.4186 - learning_rate: 0.0010
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.6000 - loss: 0.9133 - val_accuracy: 0.0000e+00 - val_loss: 3.4928 - learning_rate: 5.0000e-04
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 0.6000 - loss: 0.9844 - val_accuracy: 0.0000e+00 - val_loss: 3.5652 - learning_rate: 5.0000e-04
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.8000 - loss: 1.035

In [42]:

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
import unicodedata

BASE = Path("proyecto_documentos")
ENTRENAMIENTO = BASE / "entrenamiento"
DOCUMENTOS = BASE / "documentos"
RESULTADOS = BASE / "resultados"

RESULTADOS.mkdir(parents=True, exist_ok=True)

EXTENSIONES = {
    ".pdf", ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

def limpiar_para_modelo(texto):
    texto = unicodedata.normalize(
        "NFKD", texto.lower()
    )
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    return re.sub(r"\s+", " ", texto).strip()

# Palabras características de cada categoría.
# Se pueden ampliar con documentos reales.
PISTAS = {
    "Facturas": {
        "factura": 4,
        "factura de venta": 6,
        "numero de factura": 4,
        "subtotal": 2,
        "iva": 1
    },
    "Recibos": {
        "recibo": 5,
        "recibo de caja": 6,
        "recibi de": 4,
        "recibi la cantidad": 5
    },
    "Contratos": {
        "contrato": 5,
        "clausula": 3,
        "contratante": 3,
        "contratista": 3
    },
    "Ordenes_compra": {
        "orden de compra": 7,
        "orden de pedido": 5,
        "purchase order": 7,
        "numero de orden": 3
    }
}

plantillas_textos = []
plantillas_etiquetas = []
plantillas_archivos = []

print("LEYENDO PLANTILLAS...\n")

for carpeta in sorted(ENTRENAMIENTO.iterdir()):

    if not carpeta.is_dir():
        continue

    categoria = carpeta.name

    for archivo in sorted(carpeta.rglob("*")):

        if not archivo.is_file():
            continue

        if archivo.suffix.lower() not in EXTENSIONES:
            continue

        try:
            paginas, avisos = leer_documento(archivo)

            texto = limpiar_para_modelo(
                "\n".join(p["texto"] for p in paginas)
            )

            if len(texto) < 20:
                print("Sin suficiente texto:", archivo.name)
                continue

            plantillas_textos.append(texto)
            plantillas_etiquetas.append(categoria)
            plantillas_archivos.append(str(archivo))

            print(f"OK: {archivo.name} -> {categoria}")

        except Exception as e:
            print("Error:", archivo.name, str(e))

if not plantillas_textos:
    raise ValueError(
        "No se pudo leer texto de ninguna plantilla. "
        "Comprueba que el OCR esté instalado."
    )

vectorizador = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    sublinear_tf=True,
    max_features=20000
)

matriz_plantillas = vectorizador.fit_transform(
    plantillas_textos
)

categorias = sorted(set(plantillas_etiquetas))

def puntuacion_palabras(texto, categoria):
    pistas = PISTAS.get(categoria, {})

    puntos = 0

    for palabra, peso in pistas.items():
        patron = r"(?<!\w)" + re.escape(palabra) + r"(?!\w)"

        if re.search(patron, texto):
            puntos += peso

    return min(puntos / 10.0, 1.0)


def identificar_documento(texto):
    texto = limpiar_para_modelo(texto)

    if len(texto) < 20:
        return {
            "categoria": "Revision_necesaria",
            "coincidencia": 0.0,
            "estado": "Sin texto suficiente",
            "plantilla_similar": ""
        }

    vector = vectorizador.transform([texto])

    similitudes = cosine_similarity(
        vector, matriz_plantillas
    )[0]

    candidatos = []

    for categoria in categorias:

        indices = [
            i for i, etiqueta
            in enumerate(plantillas_etiquetas)
            if etiqueta == categoria
        ]

        mejor_indice = max(
            indices,
            key=lambda i: similitudes[i]
        )

        similitud = float(similitudes[mejor_indice])

        pistas = puntuacion_palabras(
            texto, categoria
        )

        puntuacion = (
            0.65 * similitud +
            0.35 * pistas
        )

        candidatos.append({
            "categoria": categoria,
            "puntuacion": puntuacion,
            "plantilla": Path(
                plantillas_archivos[mejor_indice]
            ).name
        })

    candidatos.sort(
        key=lambda x: x["puntuacion"],
        reverse=True
    )

    mejor = candidatos[0]

    segunda = (
        candidatos[1]["puntuacion"]
        if len(candidatos) > 1 else 0
    )

    margen = mejor["puntuacion"] - segunda

    # Umbrales iniciales de revisión.
    # No equivalen a una precisión validada.
    if mejor["puntuacion"] < 0.22 or margen < 0.07:
        categoria_final = "Revision_necesaria"
        estado = "Revisar manualmente"
    else:
        categoria_final = mejor["categoria"]
        estado = "Clasificado provisionalmente"

    return {
        "categoria": categoria_final,
        "coincidencia": round(
            mejor["puntuacion"] * 100, 1
        ),
        "estado": estado,
        "plantilla_similar": mejor["plantilla"]
    }

print("\nPLANTILLAS LEÍDAS:", len(plantillas_textos))
print("CATEGORÍAS:", categorias)


LEYENDO PLANTILLAS...

OK: 566w-7tfxbmQkCPc.png -> Comprobante
OK: Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2).pdf -> Contratos
OK: Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2)_pagina_1.png -> Contratos
OK: Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg -> Facturas
OK: image.png -> Formulario
OK: orden-de-compra-formato.png -> Ordenes_compra
OK: modelo-de-recibo-es-puro-750px.png -> Recibos

PLANTILLAS LEÍDAS: 7
CATEGORÍAS: ['Comprobante', 'Contratos', 'Facturas', 'Formulario', 'Ordenes_compra', 'Recibos']


In [43]:
import pandas as pd
import shutil
import hashlib
from pathlib import Path
from IPython.display import display

CARPETA_CLASIFICADOS = RESULTADOS / "clasificados"
CARPETA_CLASIFICADOS.mkdir(
    parents=True, exist_ok=True
)

def nombre_copia(archivo, ruta_relativa):
    # Identificador estable para evitar que archivos
    # con el mismo nombre se sobrescriban.
    identificador = hashlib.sha256(
        ruta_relativa.encode("utf-8")
    ).hexdigest()[:10]

    return (
        f"{archivo.stem}_{identificador}"
        f"{archivo.suffix}"
    )


def clasificar_todos():
    registros = []

    archivos = sorted(
        p for p in DOCUMENTOS.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTENSIONES
    )

    print("DOCUMENTOS ENCONTRADOS:", len(archivos))
    print("=" * 60)

    for numero, archivo in enumerate(archivos, 1):

        ruta_relativa = archivo.relative_to(
            DOCUMENTOS
        ).as_posix()

        try:
            paginas, avisos = leer_documento(archivo)

            texto = "\n".join(
                pagina["texto"] for pagina in paginas
            )

            resultado = identificar_documento(texto)

            categoria = resultado["categoria"]

            destino_carpeta = (
                CARPETA_CLASIFICADOS / categoria
            )
            destino_carpeta.mkdir(
                parents=True, exist_ok=True
            )

            copia = destino_carpeta / nombre_copia(
                archivo, ruta_relativa
            )

            shutil.copy2(archivo, copia)

            registro = {
                "Archivo original": archivo.name,
                "Ubicacion original": str(
                    archivo.resolve()
                ),
                "Tipo identificado": categoria,
                "Coincidencia orientativa (%)":
                    resultado["coincidencia"],
                "Estado": resultado["estado"],
                "Plantilla mas similar":
                    resultado["plantilla_similar"],
                "Paginas": len(paginas),
                "Copia clasificada": str(copia.resolve()),
                "Avisos": "; ".join(avisos)
            }

            registros.append(registro)

            print(
                f"[{numero}/{len(archivos)}] "
                f"{archivo.name}"
            )
            print("  Tipo:", categoria)
            print("  Estado:", resultado["estado"])
            print(
                "  Coincidencia:",
                resultado["coincidencia"], "%"
            )
            print("-" * 60)

        except Exception as e:
            registros.append({
                "Archivo original": archivo.name,
                "Ubicacion original": str(
                    archivo.resolve()
                ),
                "Tipo identificado": "Error",
                "Coincidencia orientativa (%)": 0,
                "Estado": "Error de procesamiento",
                "Plantilla mas similar": "",
                "Paginas": 0,
                "Copia clasificada": "",
                "Avisos": str(e)
            })
            print("ERROR:", archivo.name, e)

    df_resultados = pd.DataFrame(
        registros,
        columns=[
            "Archivo original",
            "Ubicacion original",
            "Tipo identificado",
            "Coincidencia orientativa (%)",
            "Estado",
            "Plantilla mas similar",
            "Paginas",
            "Copia clasificada",
            "Avisos"
        ]
    )

    ruta_csv = RESULTADOS / "clasificacion_final.csv"

    df_resultados.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig"
    )

    try:
        ruta_excel = (
            RESULTADOS / "clasificacion_final.xlsx"
        )

        df_resultados.to_excel(
            ruta_excel,
            index=False
        )

        print("\nExcel guardado:", ruta_excel.resolve())

    except ImportError:
        print(
            "\nPara generar Excel instala openpyxl. "
            "El CSV ya está guardado."
        )

    print("CSV guardado:", ruta_csv.resolve())

    print("\nRESUMEN POR CATEGORÍA")
    if not df_resultados.empty:
        print(
            df_resultados["Tipo identificado"]
            .value_counts()
        )

    display(df_resultados)

    return df_resultados


df_clasificados = clasificar_todos()


DOCUMENTOS ENCONTRADOS: 7
[1/7] 566w-7tfxbmQkCPc.png
  Tipo: Comprobante
  Estado: Clasificado provisionalmente
  Coincidencia: 65.0 %
------------------------------------------------------------
[2/7] Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2).pdf
  Tipo: Contratos
  Estado: Clasificado provisionalmente
  Coincidencia: 82.5 %
------------------------------------------------------------
[3/7] Documento A4 Contrato de servicios Asistente virtual Profesional Azul (2)_pagina_1.png
  Tipo: Contratos
  Estado: Clasificado provisionalmente
  Coincidencia: 65.0 %
------------------------------------------------------------
[4/7] Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg
  Tipo: Facturas
  Estado: Clasificado provisionalmente
  Coincidencia: 79.0 %
------------------------------------------------------------
[5/7] image.png
  Tipo: Formulario
  Estado: Clasificado provisionalmente
  Coincidencia: 65.0 %
----------------------------------------------------

,Archivo original,Ubicacion original,Tipo identificado,Coincidencia orientativa (%),Estado,Plantilla mas similar,Paginas,Copia clasificada,Avisos
0,566w-7tfxbmQkCPc.png,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Comprobante,65.0,Clasificado provisionalmente,566w-7tfxbmQkCPc.png,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
1,Documento A4 Contrato de servicios Asistente v...,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Contratos,82.5,Clasificado provisionalmente,Documento A4 Contrato de servicios Asistente v...,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
2,Documento A4 Contrato de servicios Asistente v...,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Contratos,65.0,Clasificado provisionalmente,Documento A4 Contrato de servicios Asistente v...,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
3,Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Facturas,79.0,Clasificado provisionalmente,Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
4,image.png,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Formulario,65.0,Clasificado provisionalmente,image.png,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
5,modelo-de-recibo-es-puro-750px.png,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Recibos,82.5,Clasificado provisionalmente,modelo-de-recibo-es-puro-750px.png,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,
6,orden-de-compra-formato.png,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,Ordenes_compra,65.0,Clasificado provisionalmente,orden-de-compra-formato.png,1,C:\Users\mcgre\Desktop\yaroy\proyecto_document...,


In [44]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ruta = Path("proyecto_documentos/resultados/resumen_documentos.csv")

if ruta.exists():
    df = pd.read_csv(ruta)

    print("DOCUMENTOS ANALIZADOS:", len(df))
    print("\nRESULTADOS DE CLASIFICACIÓN:\n")

    columnas = [
        "ruta",
        "categoria",
        "estado"
    ]

    display(df[columnas])

    print("\nCANTIDAD POR CATEGORÍA:")
    print(df["categoria"].value_counts())

else:
    print("Todavía no se encontró el archivo de resultados.")
    print("Ejecuta el bloque 5 para procesar documentos.")


DOCUMENTOS ANALIZADOS: 7

RESULTADOS DE CLASIFICACIÓN:



,ruta,categoria,estado
0,566w-7tfxbmQkCPc.png,Recibo,ok
1,Documento A4 Contrato de servicios Asistente v...,Contrato,ok
2,Documento A4 Contrato de servicios Asistente v...,Revisión necesaria,ok
3,Factura-deee9d17357b444baac4daccd31e06a6.jpg.jpeg,Factura,ok
4,image.png,Formulario,ok
5,modelo-de-recibo-es-puro-750px.png,Recibo,ok
6,orden-de-compra-formato.png,Revisión necesaria,ok



CANTIDAD POR CATEGORÍA:
categoria
Recibo                2
Revisión necesaria    2
Contrato              1
Factura               1
Formulario            1
Name: count, dtype: int64
